In [1]:
from manim import *
from scipy.stats import norm
import numpy as np
import pandas as pd

In [8]:
COLOR_LSTM = "#9370DB"
COLOR_EMBED = "#FFA500"
COLOR_FUSION = "#90EE90"
COLOR_ATTN = "#00BFFF"

DATA_DIR = '/Users/macbook/Downloads/HCMUT/Assignments/AI Projects/trend-spy-bot/src/backend/models/data'

START_DATE = "2021-06-01"
END_DATE = "2022-12-31"

In [3]:
%%manim -v WARNING --fps 120 -qk DualHeadScene

from __future__ import annotations

def create_detailed_neural_block(
    layer_sizes: list[int] = [3, 5, 3], 
    height: float = 3.5, 
    width: float = 2.5,
    color: Color = BLUE, 
    label: str = "Encoder",
    dropout: float = 0.3,
    seed: int = 42
) -> VGroup:
    np.random.seed(seed)
    rect = RoundedRectangle(
        corner_radius=0.2, height=height, width=width, 
        color=color, fill_opacity=0.1, stroke_width=2
    )
    layers_group = VGroup()
    neuron_coords = []
    n_layers = len(layer_sizes)
    x_range = np.linspace(-width/2 + 0.4, width/2 - 0.4, n_layers)
    for i, n_neurons in enumerate(layer_sizes):
        layer_nodes = VGroup()
        layer_coords = []
        if n_neurons == 1:
            y_range = [0]
        else:
            y_range = np.linspace(-height/2 + 0.5, height/2 - 0.5, n_neurons)
        x_pos = x_range[i]
        for y_pos in y_range:
            pos = rect.get_center() + np.array([x_pos, y_pos, 0])
            dot = Dot(point=pos, radius=0.07, color=color)
            dot.set_stroke(color=WHITE, width=1, opacity=0.5)
            layer_nodes.add(dot)
            layer_coords.append(pos)
        layers_group.add(layer_nodes)
        neuron_coords.append(layer_coords)
    edges_group = VGroup()
    active_edges = [] 
    for i in range(len(neuron_coords) - 1):
        curr_layer = neuron_coords[i]
        next_layer = neuron_coords[i+1]
        layer_edges = VGroup()
        for p1 in curr_layer:
            for p2 in next_layer:
                if np.random.random() > dropout:
                    line = Line(p1, p2, stroke_width=1, stroke_opacity=0.3, color=color)
                    layer_edges.add(line)
                    active_edges.append(line)
        edges_group.add(layer_edges)
    stubs = VGroup()
    for p in neuron_coords[0]:
        stubs.add(Line(p + LEFT * 0.4, p, stroke_width=2, color=color, stroke_opacity=0.5))
    for p in neuron_coords[-1]:
        stubs.add(Line(p, p + RIGHT * 0.4, stroke_width=2, color=color, stroke_opacity=0.5))
    text = Text(label, font_size=20, weight=BOLD).next_to(rect, UP, buff=0.15)
    block = VGroup(rect, edges_group, stubs, layers_group, text)
    block.rect = rect
    block.neurons = layers_group
    block.all_edges_list = active_edges
    return block

def gaussian_cloud_2d(center: np.ndarray, sigma: float, color: Color, n_points: int = 40) -> VGroup:
    points = np.random.normal(loc=center[:2], scale=sigma, size=(n_points, 2))
    points_3d = np.column_stack((points, np.zeros(n_points)))
    dots = VGroup(*[Dot(point=p, radius=0.03, color=color, fill_opacity=0.6) for p in points_3d])
    return dots

def create_score_bar(value: float, max_val: float, width: float = 0.5, color: Color = ORANGE) -> VGroup:
    norm_height = (value / max_val) * 2.2
    bar = Rectangle(width=width, height=norm_height, color=color, fill_opacity=0.8, stroke_color=WHITE, stroke_width=1)
    bar.move_to(ORIGIN, aligned_edge=DOWN)
    label = MathTex(f"{value:.1f}", font_size=24).next_to(bar, UP, buff=0.1)
    return VGroup(bar, label)

def create_gauge():
    arc = Arc(radius=0.8, start_angle=PI, angle=-PI, color=WHITE, stroke_opacity=0.5)
    green_zone = Arc(radius=0.8, start_angle=PI, angle=-PI/3, color=GREEN, stroke_width=5)
    red_zone = Arc(radius=0.8, start_angle=0, angle=PI/3, color=RED, stroke_width=5)
    needle = Line(ORIGIN, UP*0.7, color=RED, stroke_width=3)
    pivot = Dot(radius=0.05, color=WHITE)
    label = Text("Loss", font_size=16).next_to(pivot, DOWN, buff=0.2)
    return VGroup(arc, green_zone, red_zone, needle, pivot, label), needle

class DualHeadScene(Scene):
    N_ITEMS = 5
    RELEVANCES = np.array([4, 2, 5, 1, 3])
    RAW_SCORES = np.array([2.3, 1.1, 3.7, 0.4, 2.9])

    def construct(self):
        self.camera.background_color = "#1e1e1e"
        input_group = VGroup(*[
            Dot(radius=0.1, color=LIGHT_GREY) for _ in range(self.N_ITEMS)
        ]).arrange(RIGHT, buff=0.3).to_edge(LEFT, buff=0.8)
        input_label = Text("Inputs (Batch)", font_size=22, color=LIGHT_GREY).next_to(input_group, DOWN)
        self.play(FadeIn(input_group, lag_ratio=0.1), Write(input_label))
        encoder = create_detailed_neural_block(
            layer_sizes=[4, 5, 4], height=3.2, width=2.2, label="Backbone", color=TEAL
        )
        encoder.next_to(input_group, RIGHT, buff=1.5)
        connectors = VGroup(*[
            Line(d.get_right(), encoder.rect.get_left(), stroke_width=1, stroke_opacity=0.5, color=GREY)
            for d in input_group
        ])
        self.play(FadeIn(encoder), Create(connectors))
        self.play_network_pulse(encoder)
        head_buff = 3.2
        prob_center = encoder.rect.get_center() + RIGHT * head_buff + UP * 1.6
        rank_center = encoder.rect.get_center() + RIGHT * head_buff + DOWN * 1.6
        prob_box = Rectangle(width=3.2, height=2.5, color=BLUE, fill_opacity=0.05).move_to(prob_center)
        rank_box = Rectangle(width=3.2, height=2.5, color=ORANGE, fill_opacity=0.05).move_to(rank_center)
        prob_lbl = Text("Uncertainty Head", font_size=18, color=BLUE).next_to(prob_box, UP)
        rank_lbl = Text("Ranking Head", font_size=18, color=ORANGE).next_to(rank_box, DOWN)
        split_node = Dot(encoder.rect.get_right(), color=WHITE)
        path_top = CubicBezier(split_node.get_center(), split_node.get_center()+RIGHT, prob_box.get_left()+LEFT, prob_box.get_left(), color=BLUE)
        path_bot = CubicBezier(split_node.get_center(), split_node.get_center()+RIGHT, rank_box.get_left()+LEFT, rank_box.get_left(), color=ORANGE)
        self.play(
            Create(prob_box), Write(prob_lbl), Create(path_top),
            Create(rank_box), Write(rank_lbl), Create(path_bot),
            FadeIn(split_node)
        )
        mus = [np.array([-0.8, 0.4, 0]), np.array([0, 0.6, 0]), np.array([0.8, -0.4, 0]), np.array([-0.4, -0.6, 0]), np.array([0.6, 0.4, 0])]
        clouds = VGroup()
        for i, pos in enumerate(mus):
            abs_pos = prob_box.get_center() + pos
            cloud = gaussian_cloud_2d(abs_pos, 0.15, BLUE_B)
            clouds.add(cloud)
        self.play(FadeIn(clouds, lag_ratio=0.1))
        bars = VGroup()
        base_line = rank_box.get_bottom() + UP * 0.2
        start_x = rank_box.get_left()[0] + 0.4
        spacing = 0.6
        for i, score in enumerate(self.RAW_SCORES):
            b = create_score_bar(score, 4.0, color=ORANGE)
            b.move_to(np.array([start_x + i*spacing, base_line[1], 0]), aligned_edge=DOWN)
            bars.add(b)   
        self.play(LaggedStart(*[GrowFromEdge(b, DOWN) for b in bars], lag_ratio=0.1))
        springs = VGroup()
        for i in range(self.N_ITEMS):
            for j in range(i+1, self.N_ITEMS):
                delta_y = np.sign(self.RELEVANCES[i] - self.RELEVANCES[j])
                score_diff = self.RAW_SCORES[i] - self.RAW_SCORES[j]
                if (score_diff * delta_y) < 0: 
                    p1, p2 = bars[i][0].get_top(), bars[j][0].get_top()
                    spring = Line(p1, p2, color=RED, path_arc=0.5)
                    spring_zigzag = ParametricFunction(
                        lambda t: spring.point_from_proportion(t) + np.array([0, 0.05*np.sin(t*20), 0]),
                        t_range=[0, 1], color=RED
                    )
                    springs.add(spring_zigzag)
        gauge, needle = create_gauge()
        gauge.to_corner(DR, buff=0.5)
        needle.rotate(0, about_point=gauge[4].get_center()) 
        conflict_txt = Text("Order Violations", color=RED, font_size=16).next_to(rank_box, DOWN)
        if len(springs) > 0:
            self.play(Create(springs), FadeIn(conflict_txt), FadeIn(gauge))
            self.play(
                springs.animate.shift(RIGHT*0.05),
                Rotate(needle, angle=-60*DEGREES, about_point=gauge[4].get_center()),
                rate_func=wiggle, 
                run_time=1.0
            )
        sort_idx = np.argsort(-self.RAW_SCORES)
        sorted_x = [bars[i].get_x() for i in range(self.N_ITEMS)]
        anims = []
        for rank_i, orig_i in enumerate(sort_idx):
            anims.append(bars[orig_i].animate.set_x(sorted_x[rank_i]))
        self.play(
            FadeOut(springs), FadeOut(conflict_txt),
            *anims,
            Rotate(needle, angle=120*DEGREES, about_point=gauge[4].get_center()),
            run_time=2.0,
            rate_func=smooth
        )
        bridge = DashedLine(prob_box.get_bottom(), rank_box.get_top(), color=PURPLE)
        calib_txt = Text("Calibrated Ranking", font_size=24, color=WHITE).next_to(bridge, RIGHT)
        self.play(Create(bridge), Write(calib_txt))
        self.wait(2)

    def play_network_pulse(self, encoder_group):
        pulses = VGroup()
        edges = encoder_group.all_edges_list
        np.random.shuffle(edges)
        subset = edges[:15]
        for edge in subset:
            dot = Dot(radius=0.04, color=YELLOW)
            dot.move_to(edge.get_start())
            pulses.add(dot)
        self.add(pulses)
        anims = [MoveAlongPath(dot, edge, run_time=0.5, rate_func=linear) for dot, edge in zip(pulses, subset)]
        self.play(LaggedStart(*anims, lag_ratio=0.05), run_time=1.0)
        self.remove(pulses)

Manim Community v0.19.1

In [31]:
%%manim -v WARNING --fps 120 --disable_caching -qk Scene1_RealOHLCV

class Scene1_RealOHLCV(MovingCameraScene):
    def construct(self):
        df_norm = self.load_data(START_DATE, END_DATE)
        
        if df_norm is None:
            return

        y_min = df_norm.min().min()
        y_max = df_norm.max().max()
        
        y_axis_max = np.ceil(y_max * 10) / 10 + 0.1
        y_axis_min = np.floor(y_min * 10) / 10 - 0.1
        
        num_days = len(df_norm)
        
        axes = Axes(
            x_range=[0, num_days, max(1, num_days // ((pd.to_datetime(END_DATE).year - pd.to_datetime(START_DATE).year) * 12 + pd.to_datetime(END_DATE).month - pd.to_datetime(START_DATE).month + 1))],
            y_range=[y_axis_min, y_axis_max, 0.1],
            x_length=11,
            y_length=7,
            axis_config={"include_tip": False, "color": GRAY},
            y_axis_config={
                "include_numbers": True, 
                "font_size": 17,
                "decimal_number_config": {"num_decimal_places": 1},
            }
        ).center()
        
        labels = axes.get_axis_labels(
            x_label=Text("Trading Days", font_size=22), 
            y_label=Text("Return (%)", font_size=20)
        )

        stock_config = {
            "SPY": {"color": WHITE, "dashed": False},
            "AAPL": {"color": BLUE, "dashed": False},
            "MSFT": {"color": GREEN, "dashed": False},
            "GOOGL": {"color": YELLOW, "dashed": False},
            "META": {"color": RED, "dashed": False}
        }

        lines = []
        text_labels = []
        final_tags = []
        
        line_anims = []
        label_anims = []
        tag_anims = []
        fadeout_tags = []

        for ticker, style in stock_config.items():
            points = [axes.c2p(i, row[ticker]) for i, row in df_norm.iterrows()]
            
            line = VMobject().set_points_as_corners(points).set_color(style["color"]).set_stroke(width=2)
            
            label = Text(ticker, font_size=18, color=style["color"]).next_to(points[-1], RIGHT)
            
            final_val = df_norm[ticker].iloc[-1]
            tag_text = f"{final_val*100:+.1f}\%"
            tag = MathTex(tag_text, color=style["color"], font_size=24).next_to(label, RIGHT)

            lines.append(line)
            text_labels.append(label)
            final_tags.append(tag)
            
            line_anims.append(Create(line))
            label_anims.append(FadeIn(label))
            tag_anims.append(Write(tag))
            fadeout_tags.append(FadeOut(tag))

        self.play(Create(axes), Write(labels))
        self.wait(0.7)
        
        self.play(
            *line_anims,
            run_time=3,
            rate_func=linear
        )
        
        self.play(FadeOut(labels), run_time=0.5)
        self.play(*label_anims)
        
        self.remove(labels)
        
        zoom_center = axes.c2p(num_days, (y_max + y_min)/2)
        self.play(
            self.camera.frame.animate.scale(0.5).move_to(zoom_center),
            run_time=2
        )
        
        self.play(*tag_anims)
        
        self.wait(1)
        self.remove(*fadeout_tags)
        
        self.play(
            FadeOut(axes), 
            *[FadeOut(obj) for obj in lines + text_labels + final_tags],
            self.camera.frame.animate.scale(2).move_to(ORIGIN),
            run_time=1.5
        )
        
        final_stats = []
        for ticker, style in stock_config.items():
            final_stats.append({
                "name": ticker,
                "ret": df_norm[ticker].iloc[-1],
                "color": style["color"]
            })
        final_stats.sort(key=lambda x: x["ret"], reverse=True)
        
        title = Text("Relative Performance Ranking", font_size=40).to_edge(UP)
        self.play(Write(title), run_time=1)

        for i, item in enumerate(final_stats):
            rank = Text(f"#{i+1}", font_size=36, color=GRAY).shift(UP*(2 - i*1.0) + LEFT*3)
            name = Text(item["name"], font_size=36, color=item["color"]).next_to(rank, RIGHT, buff=1)
            val_text = f"{item['ret']*100:+.1f}%"
            val = Text(val_text, font_size=36, color=item["color"]).next_to(name, RIGHT, buff=1)

            anims = [FadeIn(rank), FadeIn(name), FadeIn(val)]

            if i == 0:
                winner_group = VGroup(rank, name, val)
                box = SurroundingRectangle(winner_group, color=YELLOW, buff=0.06)
                alpha_txt = Text("ALPHA LEADER", font_size=20, color=YELLOW).next_to(box, UP)
                anims.extend([Create(box), FadeIn(alpha_txt)])
            
            self.play(*anims, run_time=0.6)
        self.wait(1)

    def load_data(self, start, end):
        tickers = ['SPY', 'AAPL', 'MSFT', 'GOOGL', 'META']
        
        try:
            dfs = []
            for t in tickers:
                file_path = f"{DATA_DIR}/{t}.csv"
                df = pd.read_csv(file_path, parse_dates=['Date'])
                
                price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
                
                df = df[['Date', price_col]].rename(columns={price_col: t})
                dfs.append(df)
            
            df_merged = dfs[0]
            for i in range(1, len(dfs)):
                df_merged = df_merged.merge(dfs[i], on='Date', how='inner')
            
            mask = (df_merged['Date'] >= start) & (df_merged['Date'] <= end)
            df_final = df_merged.loc[mask].copy()
            
            if df_final.empty:
                print(f"Error: No data found between {start} and {end}.")
                return None
            
            df_final = df_final.reset_index(drop=True)
            
            cols = tickers
            initial_prices = df_final.iloc[0][cols]
            df_norm = df_final[cols] / initial_prices - 1
            
            return df_norm

        except Exception as e:
            print(f"Data Loading Error: {e}")
            print("Generating DUMMY data for visualization purposes...")
            dates = pd.date_range(start=start, end=end, freq='D')
            
            data = {}
            data['SPY'] = np.cumsum(np.random.normal(0.0003, 0.01, size=len(dates)))
            data['AAPL'] = np.cumsum(np.random.normal(0.0005, 0.015, size=len(dates)))
            data['GOOGL'] = np.cumsum(np.random.normal(0.0004, 0.014, size=len(dates)))
            data['META'] = np.cumsum(np.random.normal(0.0006, 0.020, size=len(dates)))
            
            return pd.DataFrame(data, index=dates).reset_index(drop=True)

<string>:60: SyntaxWarning: invalid escape sequence '\%'


Manim Community v0.19.1

In [32]:
%%manim -v WARNING --fps 120 --disable_caching -qk Scene2_SecretSauceReal

class Scene2_SecretSauceReal(Scene):
    def construct(self):
        day_data, date_str = self.get_worst_day_data()
        if day_data is None: return

        tickers = list(day_data.keys())
        raw_values = np.array(list(day_data.values()))
        
        if np.mean(np.abs(raw_values)) > 1.0:
            print("Detected values > 1.0, assuming percentage integers. Dividing by 100.")
            raw_values = raw_values / 100.0

        mean_ret = np.mean(raw_values)
        
        data_min = raw_values.min()
        data_max = raw_values.max()
        
        y_min_limit = min(data_min, -0.05) * 1.3
        y_max_limit = max(data_max, 0.05) * 1.3

        y_min_limit = np.floor(y_min_limit * 100) / 100
        y_max_limit = np.ceil(y_max_limit * 100) / 100

        raw_range = y_max_limit - y_min_limit
        if raw_range > 0.2:
            tick_step = 0.05
        elif raw_range > 0.1:
            tick_step = 0.02
        else:
            tick_step = 0.01

        axes = Axes(
            x_range=[0, len(tickers) + 1, 1],
            y_range=[y_min_limit, y_max_limit, tick_step],
            x_length=10,
            y_length=5.0,
            axis_config={
                "include_tip": False,
                "color": GRAY
            },
            y_axis_config={
                "include_numbers": True,
                "decimal_number_config": {
                    "num_decimal_places": 2
                }
            }
        ).center().shift(DOWN * 0.3)
        
        zero_line = Line(axes.c2p(0, 0), axes.c2p(len(tickers)+1, 0), color=WHITE, stroke_width=2)
        y_lbl = axes.get_y_axis_label(Text("Return", font_size=36)).scale(0.7).next_to(axes.y_axis, UP)

        title = Text(f"Market Crash Analysis: {date_str}", font_size=36).to_edge(UP)
        self.play(Write(title))
        
        bars = VGroup()
        labels = VGroup()
        val_texts = VGroup()
        BAR_WIDTH = 0.8

        for i, val in enumerate(raw_values):
            p_origin = axes.c2p(i+1, 0)
            p_val = axes.c2p(i+1, val)
            bar_height = abs(p_val[1] - p_origin[1])
            
            bar = Rectangle(
                height=bar_height,
                width=BAR_WIDTH,
                fill_color=RED,
                fill_opacity=0.9,
                stroke_width=0
            )
            
            if val < 0:
                bar.move_to(p_origin, aligned_edge=UP)
                lbl = Text(tickers[i], font_size=20).next_to(p_origin, UP, buff=0.2)
                vt = Text(f"{val*100:.1f}%", font_size=18, color=RED).next_to(bar, DOWN, buff=0.1)
            else:
                bar.move_to(p_origin, aligned_edge=DOWN)
                lbl = Text(tickers[i], font_size=20).next_to(p_origin, DOWN, buff=0.2)
                vt = Text(f"{val*100:.1f}%", font_size=18, color=RED).next_to(bar, UP, buff=0.1)

            bars.add(bar)
            labels.add(lbl)
            val_texts.add(vt)

        self.play(Create(axes), Create(zero_line), FadeIn(y_lbl))
        self.play(
            LaggedStart(*[GrowFromEdge(b, UP) for b in bars], lag_ratio=0.25),
            FadeIn(labels),
            FadeIn(val_texts),
            run_time=2
        )
        self.wait(1)

        mean_y = axes.c2p(0, mean_ret)[1]
        mean_line = DashedLine(
            start=[axes.c2p(0, 0)[0], mean_y, 0],
            end=[axes.c2p(len(tickers)+1, 0)[0], mean_y, 0],
            color=YELLOW,
            stroke_width=3
        )
        
        mean_text_str = f"Mean: {mean_ret*100:.1f}%"
        mean_lbl = Text(mean_text_str, color=YELLOW, font_size=24)
        mean_lbl_bg = BackgroundRectangle(mean_lbl, color=BLACK, fill_opacity=0.7, buff=0.1)
        mean_group = VGroup(mean_lbl_bg, mean_lbl).next_to(mean_line, RIGHT, aligned_edge=RIGHT)
        
        self.play(Create(mean_line), FadeIn(mean_group))
        self.wait(1)

        formula = MathTex(
            r"z_{i,t} = \frac{x_{i,t} - \mu_t}{\sigma_t}",
            font_size=40
        ).to_edge(UP).shift(DOWN*0.5)
        formula.set_color_by_tex("x_{i,t}", RED)
        formula.set_color_by_tex("\\mu_t", YELLOW)
        
        self.play(FadeOut(title), FadeIn(formula))
        
        shifted_values = raw_values - mean_ret 
        
        new_bars = VGroup()
        new_val_texts = VGroup()
        new_labels = []

        for i, val in enumerate(shifted_values):
            p_origin = axes.c2p(i+1, 0)
            p_val = axes.c2p(i+1, val)
            bar_height = abs(p_val[1] - p_origin[1])
            
            new_bar = Rectangle(
                height=bar_height,
                width=BAR_WIDTH,
                fill_color=GREEN if val >= 0 else RED, 
                fill_opacity=0.9,
                stroke_width=0
            )
            
            if val >= 0:
                new_bar.move_to(p_origin, aligned_edge=DOWN)
                nvt_pos = UP
                lbl_pos = DOWN
            else:
                new_bar.move_to(p_origin, aligned_edge=UP)
                nvt_pos = DOWN
                lbl_pos = UP
            
            # Name label
            new_lbl = Text(tickers[i], font_size=20)
            new_lbl.next_to(new_bar, lbl_pos, buff=0.2)
            new_labels.append(new_lbl)

            new_bars.add(new_bar)
            nvt_str = f"{val*100:+.1f}%"
            nvt_color = RED if val < 0 else GREEN
            nvt = Text(nvt_str, font_size=18, color=nvt_color)
            nvt.next_to(new_bar, nvt_pos, buff=0.1)
            new_val_texts.add(nvt)

        self.play(
            Indicate(formula),
            FadeOut(y_lbl),
            AnimationGroup(
                mean_line.animate.move_to(zero_line.get_center()),
                mean_group.animate.next_to(zero_line, RIGHT, buff=0.1).set_opacity(0),
                lag_ratio=0,
            ),
            Transform(bars, new_bars),
            FadeOut(val_texts),
            ReplacementTransform(labels, VGroup(*new_labels)),
            run_time=3
        )
        self.play(FadeIn(new_val_texts))
        
        winner_idx = np.argmax(shifted_values)
        loser_idx = np.argmin(shifted_values)
        
        win_group = VGroup(bars[winner_idx], new_val_texts[winner_idx])
        lose_group = VGroup(bars[loser_idx], new_val_texts[loser_idx])
        
        brace_win = Brace(win_group, UP, color=GREEN, buff=0.1)
        lbl_win = brace_win.get_text("Alpha").set_color(GREEN).scale(0.8)
        
        brace_lose = Brace(lose_group, DOWN, color=RED, buff=0.1)
        lbl_lose = brace_lose.get_text("Beta Drag").set_color(RED).scale(0.8)
        
        self.play(
            GrowFromCenter(brace_win), FadeIn(lbl_win),
            GrowFromCenter(brace_lose), FadeIn(lbl_lose),
            formula.animate.to_edge(DOWN, buff=0.6)
        )
        self.play(
            Create(SurroundingRectangle(formula, color=YELLOW, buff=0.15)),
            Indicate(formula),
            run_time = 1.5)
        
        self.wait(0.5)

    def get_worst_day_data(self):
        tickers = ['SPY', 'AAPL', 'NVDA', 'GOOGL', 'META']
        try:
            dfs = []
            for t in tickers:
                df = pd.read_csv(f"{DATA_DIR}/{t}.csv", parse_dates=['Date'])
                price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
                df = df[['Date', price_col]].rename(columns={price_col: t})
                dfs.append(df)
            
            df_merged = dfs[0]
            for i in range(1, len(dfs)):
                df_merged = df_merged.merge(dfs[i], on='Date', how='inner')
            
            mask = (df_merged['Date'] >= START_DATE) & (df_merged['Date'] <= END_DATE)
            df_filtered = df_merged.loc[mask].set_index('Date')
            df_rets = df_filtered.pct_change().dropna()
            
            df_rets['mean'] = df_rets.mean(axis=1)
            worst_date = df_rets['mean'].idxmin()
            worst_row = df_rets.loc[worst_date].drop('mean')
            
            print(f"Worst Day: {worst_date.date()} ({df_rets.loc[worst_date]['mean']:.2%})")
            return worst_row.to_dict(), str(worst_date.date())
            
        except Exception as e:
            print(f"Error: {e}")
            return None, None

Manim Community v0.19.1

Worst Day: 2022-02-03 (-7.77%)


In [ ]:
%%manim -v WARNING --fps 120 --disable_caching -qk LogReturn

from manim import *
import pandas as pd
import numpy as np

DATA_DIR = '/Users/macbook/Downloads/HCMUT/Assignments/AI Projects/trend-spy-bot/src/backend/models/data'
START = "2022-01-01"
END = "2022-04-01"

class LogReturn(MovingCameraScene):
    def construct(self):
        df_norm = self.load_data(START, END)
        if df_norm is None:
            return

        ticker = 'NVDA'
        steps = 40
        steps = min(steps, len(df_norm) - 1)

        subset = df_norm[ticker].iloc[:steps+1].values
        prices = (subset + 1) * 100
        log_returns = [np.log(prices[i]/prices[i-1]) for i in range(1, len(prices))]

        title = Text(f"Feature Engineering: Log Returns ({ticker})", font_size=36).to_edge(UP)

        formula = MathTex(
            r"{r_t} = \ln\left(\frac{P_t}{P_{t-1}}\right) \approx \frac{P_t - P_{t-1}}{P_{t-1}}",
            font_size=28,
            color=YELLOW
        ).next_to(title, DOWN, buff=0.3)

        self.play(Write(title), FadeIn(formula, shift=DOWN), run_time=1.5)

        p_min, p_max = min(prices), max(prices)
        y_min = np.floor(p_min / 5) * 5
        y_max = np.ceil(p_max / 5) * 5
        
        price_axes = Axes(
            x_range=[0, steps, 5], 
            y_range=[y_min, y_max, (y_max-y_min)/4], 
            x_length=10, 
            y_length=2.5,
            axis_config={"include_tip": False, "color": GRAY},
            y_axis_config={
                "include_numbers": True, 
                "font_size": 17,
                "decimal_number_config": {"num_decimal_places": 0}
            }
        ).shift(UP * 0.5)
        
        price_labels = price_axes.get_axis_labels(
            x_label=Text("Days", font_size=16), 
            y_label=Text("Price ($)", font_size=16)
        )

        points = [price_axes.c2p(i, p) for i, p in enumerate(prices)]
        price_line = VMobject().set_points_as_corners(points).set_color(RED_C).set_stroke(width=2)
        
        ticker_lbl = Text(ticker, font_size=18, color=BLUE).next_to(points[-1], RIGHT)
        
        final_val = prices[-1]
        tag_text = f"${final_val:.0f}"
        tag = Text(tag_text, color=BLUE, font_size=18).next_to(ticker_lbl, RIGHT)

        self.play(Create(price_axes), Write(price_labels))
        self.wait(0.5)
        
        self.play(
            Create(price_line), 
            FadeIn(ticker_lbl),
            Write(tag),
            run_time=2.5, 
            rate_func=linear
        )

        trend_arrow = DashedLine(
            price_axes.c2p(0, prices[0]), 
            price_axes.c2p(steps, prices[-1]), 
            color=GRAY, stroke_opacity=0.5
        )
        trend_line_mid = (price_axes.c2p(0, prices[0]) + price_axes.c2p(steps, prices[-1])) / 2
        angle = (price_axes.c2p(steps, prices[-1]) - price_axes.c2p(0, prices[0]))
        label_angle = np.arctan2(angle[1], angle[0])
        trend_txt = Text("Non-Stationary Trend", font_size=16, color=RED)\
            .move_to(trend_line_mid + np.array([0.1, 0.25, 0]))\
            .rotate(label_angle)
        
        self.play(FadeIn(trend_arrow), FadeIn(trend_txt))
        self.wait(0.5)

        top_group = VGroup(price_axes, price_labels, price_line, ticker_lbl, tag, trend_arrow, trend_txt)

        self.play(
            FadeOut(title),
            formula.animate.scale(0.9).to_edge(UP, buff=0.2),
            top_group.animate.shift(UP * 1.2),
            run_time=1.5
        )

        if len(log_returns) == 0:
            r_max = 0.02
        else:
            r_max = max(np.abs(log_returns)) * 1.2
            r_max = np.ceil(r_max * 100) / 100

        ret_axes = Axes(
            x_range=[0, steps, 5],
            y_range=[-r_max, r_max, r_max/2 if r_max > 0 else 0.01],
            x_length=10,
            y_length=2.5,
            axis_config={"include_tip": False, "color": GRAY},
            y_axis_config={"include_numbers": True, "decimal_number_config": {"num_decimal_places": 2}}
        ).next_to(price_axes, DOWN, buff=0.5)

        ret_lbl = MathTex(r"\text{Momentum}\ (r_t)", font_size=20, color=YELLOW).next_to(ret_axes, UP, buff=0.1, aligned_edge=LEFT)
        zero_line = DashedLine(ret_axes.c2p(0, 0), ret_axes.c2p(steps, 0), color=WHITE, stroke_opacity=0.5)

        self.play(Create(ret_axes), Write(ret_lbl), Create(zero_line))

        bars = VGroup()
        self.camera.frame.save_state()

        for i in range(1, steps+1):
            val = log_returns[i-1]

            p1 = price_axes.c2p(i-1, prices[i-1])
            p2 = price_axes.c2p(i, prices[i])

            segment = Line(p1, p2, color=YELLOW, stroke_width=4)

            color = GREEN if val > 0 else RED
            p_zero = ret_axes.c2p(i, 0)
            p_val = ret_axes.c2p(i, val)
            bar_height = abs(p_val[1] - p_zero[1])

            bar = Rectangle(width=0.2, height=bar_height, fill_color=color, fill_opacity=0.9, stroke_width=0)
            bar.move_to(p_zero, aligned_edge=DOWN if val > 0 else UP)
            bars.add(bar)

            if i <= 5:
                focus_group = VGroup(segment, bar)
                target_point = focus_group.get_center()

                if i == 1:
                    self.play(
                        self.camera.frame.animate.scale(0.7).move_to(target_point),
                        run_time=1.0, rate_func=smooth
                    )
                else:
                    self.play(
                        self.camera.frame.animate.move_to(target_point), 
                        run_time=0.4
                    )

                self.play(Create(segment), run_time=0.15)
                self.play(TransformFromCopy(segment, bar), run_time=0.25,
                          FadeOut(segment), run_time=0.1)

            elif i == 6:
                self.play(Restore(self.camera.frame), run_time=1.2, rate_func=smooth)
                self.add(bar)
            else:
                lag_ratio = abs(prices[i] / prices[i-1] - 1)
                self.play(Create(bar), run_time=0.07)
                if lag_ratio > 0.01:
                    self.wait(0.01)

        box = SurroundingRectangle(bars, color=YELLOW, buff=0.1, stroke_opacity=0.5)
        stat_txt = Text("Stationary Features (Mean ≈ 0)", font_size=24, color=YELLOW).next_to(box, DOWN)

        self.play(Create(box), Write(stat_txt))
        self.play(FadeOut(trend_arrow), FadeOut(trend_txt))
        self.wait(2)

    def load_data(self, start, end):
        tickers = ['NVDA']
        try:
            dfs = []
            for t in tickers:
                file_path = f"{DATA_DIR}/{t}.csv"
                df = pd.read_csv(file_path, parse_dates=['Date'])
                if df.empty:
                    raise ValueError(f"CSV file {file_path} is empty.")
                price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
                df = df[['Date', price_col]].rename(columns={price_col: t})
                dfs.append(df)
            df_merged = dfs[0]
            for i in range(1, len(dfs)):
                df_merged = df_merged.merge(dfs[i], on='Date', how='inner')

            mask = (df_merged['Date'] >= start) & (df_merged['Date'] <= end)
            df_final = df_merged.loc[mask].copy()

            if df_final.empty:
                print(f"Error: No data found between {start} and {end}.")
                return None

            df_final = df_final.reset_index(drop=True)
            cols = tickers
            initial_prices = df_final.iloc[0][cols]
            df_norm = df_final[cols] / initial_prices - 1
            print(f"Successfully loaded {tickers[0]} data.")
            return df_norm

        except Exception as e:
            print(f"Data Loading Error: {e}")
            print("Generating DUMMY data for visualization purposes...")
            dates = pd.date_range(start=start, end=end, freq='D')
            data = {t: np.cumsum(np.random.normal(0.0005, 0.015, size=len(dates))) for t in tickers}
            return pd.DataFrame(data, index=dates).reset_index(drop=True)

Manim Community v0.19.1

Successfully loaded NVDA data.


In [6]:
%%manim -v WARNING --fps 120 --disable_caching -qk SMADistanceScene

def load_data(start, end):
    tickers = ['NVDA']
    try:
        dfs = []
        for t in tickers:
            file_path = f"{DATA_DIR}/{t}.csv"
            df = pd.read_csv(file_path, parse_dates=['Date'])
            if df.empty:
                raise ValueError(f"CSV file {file_path} is empty.")
            price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
            df = df[['Date', price_col]].rename(columns={price_col: t})
            dfs.append(df)

        df_merged = dfs[0]
        for i in range(1, len(dfs)):
            df_merged = df_merged.merge(dfs[i], on='Date', how='inner')

        mask = (df_merged['Date'] >= start) & (df_merged['Date'] <= end)
        df_final = df_merged.loc[mask].copy()

        if df_final.empty:
            print(f"Error: No data found between {start} and {end}.")
            return None

        df_final = df_final.reset_index(drop=True)
        cols = tickers
        initial_prices = df_final.iloc[0][cols]
        df_norm = df_final[cols] / initial_prices - 1
        print(f"Successfully loaded {tickers[0]} data.")
        return df_norm

    except Exception as e:
        print(f"Data Loading Error: {e}")
        print("Generating DUMMY data for visualization purposes...")
        dates = pd.date_range(start=start, end=end, freq='D')
        np.random.seed(42)
        returns = np.random.normal(0.001, 0.02, size=len(dates))
        price_path = np.cumprod(1 + returns) - 1
        data = pd.DataFrame({'Date': dates, 'NVDA': price_path})
        return data
    
class SMADistanceScene(MovingCameraScene):
    def construct(self):
        # Set up
        df_norm = load_data(START_DATE, END_DATE)
        if df_norm is None:
            return

        df = df_norm.rename(columns={'NVDA': 'Price'})
        df['SMA10'] = df['Price'].rolling(window=10).mean()
        df['SMA20'] = df['Price'].rolling(window=20).mean()
        df['SMA60'] = df['Price'].rolling(window=60).mean()

        df = df.dropna().reset_index(drop=True)
        if len(df) > 300:
            df = df.iloc[-300:].reset_index(drop=True)

        vals = df[['Price', 'SMA10', 'SMA20', 'SMA60']]
        price_min = vals.min().min()
        price_max = vals.max().max()

        y_padding = (price_max - price_min) * 0.1
        price_min -= y_padding
        price_max += y_padding
        step_y = (price_max - price_min) / 4

        # ManimCE Axes uses x_length/y_length
        axes = Axes(
            x_range=[0, len(df), 30],
            y_range=[price_min, price_max, step_y],
            x_length=12,
            y_length=6,
            axis_config={
                "include_tip": False,
                "color": GREY,
                "stroke_width": 3
            },
            y_axis_config={
                "decimal_number_config": {"num_decimal_places": 2}
            }
        ).center()

        # ManimCE method to add numbers
        axes.add_coordinates(font_size=18)

        x_label = Text("Trading Days", font_size=20).next_to(axes.x_axis, UP, buff=0.2)
        y_label = Text("Norm. Return", font_size=20).next_to(axes.y_axis, UP, buff=0.2)
        labels = VGroup(x_label, y_label)

        self.play(Create(axes), Write(labels))
        
        # Price & SMAs
        price_line = self.get_line_graph(df, 'Price', axes, WHITE, stroke_width=3)
        price_label = Text("NVDA", font_size=20, color=WHITE).next_to(price_line.get_end(), RIGHT)

        self.play(Create(price_line), run_time=2, rate_func=linear)
        self.play(FadeIn(price_label))
        self.wait(0.5)
        self.play(FadeOut(price_label))

        sma_configs = [
            ('SMA10', GREEN, r"\text{SMA}_{10}"),
            ('SMA20', YELLOW, r"\text{SMA}_{20}"),
            ('SMA60', RED, r"\text{SMA}_{60}")
        ]
        sma_lines_group = VGroup()
        sma_labels_group = VGroup()
        for col, color, txt in sma_configs:
            line = self.get_line_graph(df, col, axes, color, stroke_width=4)
            lbl = MathTex(txt, font_size=24, color=color).next_to(line.get_end(), RIGHT)
            sma_lines_group.add(line)
            sma_labels_group.add(lbl)
            self.play(
                Create(line),
                FadeIn(lbl, shift=LEFT),
                run_time=1.5
            )
            self.wait(0.5)
            self.play(FadeOut(lbl))

        self.wait(2)
            
        # Distance to SMA_60
        idx = 124
        frame = self.camera.frame
        
        price_pt = axes.c2p(idx, df.iloc[idx]['Price'])
        sma60_pt = axes.c2p(idx, df.iloc[idx]['SMA60'])
        gap_line = DashedLine(price_pt, sma60_pt, color=BLUE)
        gap_brace = Brace(gap_line, RIGHT, buff=0.05)
        gap_brace.set_color(BLUE)

        self.play(
            frame.animate.scale(0.4).move_to(gap_line),
            run_time=2,
            rate_func=smooth
        )

        self.play(Create(gap_line), Create(gap_brace))

        formula = MathTex(
            r"d_{i,t} = P_t - \text{SMA}_{i,t}",
            font_size=24,
            color=BLUE
        ).next_to(gap_brace, RIGHT, buff=0.1)

        bg_rect = BackgroundRectangle(formula, color=BLACK, fill_opacity=0.7)

        self.play(FadeIn(bg_rect), Write(formula))
        self.wait(1)

        main_chart_group = VGroup(
            axes, labels, price_line, sma_lines_group
        )

        self.play(
            FadeOut(gap_line), FadeOut(gap_brace),
            FadeOut(bg_rect), FadeOut(formula),
            run_time=1
        )

        # Oscillator
        self.play(
            frame.animate.scale(2.5).move_to(ORIGIN),
            main_chart_group.animate.scale(0.5).to_edge(UP, buff=0.8),
            run_time=2,
            rate_func=smooth
        )

        dist_values = (df['Price'] - df['SMA60'])
        max_dist = max(dist_values.abs().max(), 0.01)

        dist_axes = Axes(
            x_range=[0, len(df), 30],
            y_range=[-max_dist, max_dist, max_dist],
            x_length=10,
            y_length=2.5,
            axis_config={"include_tip": False, "color": GREY},
            y_axis_config={"decimal_number_config": {"num_decimal_places": 2}}
        )
        dist_axes.add_coordinates(font_size=14)
        dist_axes.to_edge(DOWN, buff=0.5)

        dist_label = Text("Distance Feature (Oscillator)", font_size=20, color=BLUE)
        dist_label.next_to(dist_axes, UP, aligned_edge=LEFT)

        zero_line = Line(dist_axes.c2p(0, 0), dist_axes.c2p(len(df), 0), color=WHITE, stroke_opacity=0.5)

        dist_points = [dist_axes.c2p(i, v) for i, v in enumerate(dist_values)]
        dist_curve = VMobject().set_points_as_corners(dist_points).set_color(BLUE)

        area_lines = VGroup()
        for i, val in enumerate(dist_values):
            if i % 2 == 0:
                p1 = dist_axes.c2p(i, 0)
                p2 = dist_axes.c2p(i, val)
                color = GREEN if val > 0 else RED
                line = Line(p1, p2, color=color, stroke_width=1.5, stroke_opacity=0.6)
                area_lines.add(line)

        self.play(
            Create(dist_axes),
            Write(dist_label),
            Create(zero_line),
            run_time=2
        )

        self.play(
            Create(dist_curve),
            FadeIn(area_lines),
            run_time=3
        )
        self.wait(2)
        
    def get_line_graph(self, df, col_name, axes, color, stroke_width=2):
        points = [axes.c2p(i, row[col_name]) for i, row in df.iterrows()]
        line = VMobject().set_points_as_corners(points).set_color(color).set_stroke(width=stroke_width)
        return line

Manim Community v0.19.1

Successfully loaded NVDA data.


In [10]:
%%manim -v WARNING --fps 120 --disable_caching -qk RollingVolatilityScene

from manim import *
import pandas as pd
import numpy as np
from manim.utils.rate_functions import smooth, linear

def load_data(start, end):
    tickers = ['NVDA']
    try:
        dfs = []
        for t in tickers:
            file_path = f"{DATA_DIR}/{t}.csv"
            df = pd.read_csv(file_path, parse_dates=['Date'])
            if df.empty:
                raise ValueError(f"CSV file {file_path} is empty.")
            price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
            df = df[['Date', price_col]].rename(columns={price_col: t})
            dfs.append(df)

        df_merged = dfs[0]
        for i in range(1, len(dfs)):
            df_merged = df_merged.merge(dfs[i], on='Date', how='inner')

        mask = (df_merged['Date'] >= start) & (df_merged['Date'] <= end)
        df_final = df_merged.loc[mask].copy()

        if df_final.empty:
            print(f"Error: No data found between {start} and {end}.")
            return None

        df_final = df_final.reset_index(drop=True)
        cols = tickers
        initial_prices = df_final.iloc[0][cols]
        df_norm = df_final[cols] / initial_prices - 1
        print(f"Successfully loaded {tickers[0]} data.")
        return df_norm

    except Exception as e:
        print(f"Data Loading Error: {e}")
        print("Generating DUMMY data for visualization purposes...")
        dates = pd.date_range(start=start, end=end, freq='D')
        np.random.seed(42)
        returns = np.random.normal(0.001, 0.02, size=len(dates))
        price_path = np.cumprod(1 + returns) - 1
        data = pd.DataFrame({'Date': dates, 'NVDA': price_path})
        return data

class RollingVolatilityScene(Scene):
    def construct(self):
        # 1. Prepare Data
        df_norm = load_data(START_DATE, END_DATE)
        if df_norm is None:
            return

        df = df_norm.rename(columns={'NVDA': 'Price'})
        df['Returns'] = df['Price'].pct_change()
        df['Vol10'] = df['Returns'].rolling(window=10).std()
        df['Vol20'] = df['Returns'].rolling(window=20).std()
        df['Vol60'] = df['Returns'].rolling(window=60).std()
        df = df.dropna().reset_index(drop=True)

        if len(df) > 300:
            df = df.iloc[-300:].reset_index(drop=True)

        price_min, price_max = df['Price'].min(), df['Price'].max()
        p_padding = (price_max - price_min) * 0.1

        # 2. Setup Top Axes (Price)
        axes_top = Axes(
            x_range=[0, len(df), 30],
            y_range=[price_min - p_padding, price_max + p_padding, (price_max - price_min) / 4],
            x_length=11,  # CE uses x_length instead of width
            y_length=3.0, # CE uses y_length instead of height
            axis_config={
                "color": GREY, 
                "stroke_width": 2,
                "include_tip": False
            },
            y_axis_config={
                "decimal_number_config": {"num_decimal_places": 2}
            }
        ).to_edge(UP, buff=0.5)

        # CE manual label addition
        axes_top.add_coordinates(font_size=16)

        # 3. Setup Bottom Axes (Volatility)
        vol_max = max(df['Vol10'].max(), df['Vol20'].max(), df['Vol60'].max()) * 1.1

        axes_bot = Axes(
            x_range=[0, len(df), 30],
            y_range=[0, vol_max, vol_max / 4],
            x_length=11,
            y_length=2.5,
            axis_config={
                "color": GREY, 
                "stroke_width": 2, 
                "include_tip": False
            },
            y_axis_config={
                "decimal_number_config": {"num_decimal_places": 3}
            }
        ).to_edge(DOWN, buff=0.5)

        axes_bot.add_coordinates(font_size=16)

        label_bot = Text("Rolling Volatility", font_size=20, color=ORANGE)
        label_bot.next_to(axes_bot, UP, aligned_edge=LEFT, buff=0.2)

        self.play(
            Create(axes_top),
            Create(axes_bot),
            Write(label_bot)
        )

        price_line = self.get_line_graph(df, 'Price', axes_top, WHITE, stroke_width=2)
        self.play(Create(price_line), run_time=1.5)

        # 4. Intro Formula
        formula = MathTex(
            r"\sigma_t = \sqrt{\frac{1}{N-1} \sum_{i=0}^{N-1} (r_{t-i} - \bar{r})^2}",
            font_size=32, color=YELLOW
        ).move_to(axes_bot.get_center() + UP * 2.5)
        
        concept_text = Text(
            "Standard Deviation of returns over moving windows",
            font_size=20, color=GREY_A
        ).next_to(formula, DOWN)

        self.play(Write(formula))
        self.play(FadeIn(concept_text))
        self.wait(1)
        self.play(FadeOut(concept_text))

        # 5. Rolling Windows Loop
        configs = [
            (10, GREEN, 'Vol10'),
            (20, YELLOW, 'Vol20'),
            (60, RED, 'Vol60')
        ]

        # Calculate unit width for rectangles
        x_len = axes_top.x_length
        x_range = axes_top.x_range[1] - axes_top.x_range[0]
        unit_size = x_len / x_range

        final_curves = VGroup()

        for window_size, color, col_name in configs:
            formula_colors = {10: GREEN, 20: YELLOW, 60: RED}
            
            # Specific Formula
            formula_specific = MathTex(
                rf"\sigma_{{t}}^{{({window_size})}} = \sqrt{{\frac{{1}}{{{window_size}-1}} \sum_{{i=0}}^{{{window_size}-1}} (r_{{t-i}} - \bar{{r}})^2}}",
                font_size=32, color=formula_colors.get(window_size, YELLOW)
            ).move_to(axes_bot.get_center() + UP * 2.5)
            
            # Prepare Data Points
            vol_points = [axes_bot.c2p(i, val) for i, val in enumerate(df[col_name])]
            
            # Window Rectangle
            window_width = unit_size * window_size
            window_rect = Rectangle(
                width=window_width,
                height=axes_top.y_length,
                fill_color=color,
                fill_opacity=0.2,
                stroke_width=0
            )
            
            start_x = axes_top.c2p(window_size, 0)[0]
            # Center the window rectangle (Manim rectangles are centered by default)
            window_rect.move_to(np.array([start_x - window_width/2, axes_top.get_center()[1], 0]))
            
            # Running Label
            run_label = Text(f"{window_size} Days", font_size=24, color=color)
            run_label.add_updater(lambda m: m.next_to(window_rect, DOWN))

            # Partial Curve (The "Pen" tip)
            partial_curve = VMobject().set_points_as_corners([vol_points[0], vol_points[0]])
            partial_curve.set_color(color).set_stroke(width=2)
            
            # Animation Trackers
            tracker = ValueTracker(window_size)
            total_days = len(df) - 1
            
            # --- Easing Logic ---
            def get_eased_day():
                current_linear = tracker.get_value()
                alpha = (current_linear - window_size) / max(1, (total_days - window_size))
                alpha_eased = smooth(alpha) 
                eased_day = window_size + alpha_eased * (total_days - window_size)
                return min(eased_day, total_days)

            def window_updater(mob):
                day = get_eased_day()
                x_curr = axes_top.c2p(day, 0)[0]
                mob.set_x(x_curr - window_width/2)

            def curve_updater(mob):
                day = int(get_eased_day())
                if day >= len(vol_points): day = len(vol_points)
                visible_points = vol_points[:day]
                if len(visible_points) > 1:
                    mob.set_points_as_corners(visible_points)

            window_rect.add_updater(window_updater)
            partial_curve.add_updater(curve_updater)
            
            # Start Animation
            self.play(
                FadeIn(window_rect), 
                FadeIn(run_label), 
                FadeIn(partial_curve), 
                Transform(formula, formula_specific)
            )
            
            # Run the slide
            self.play(
                tracker.animate.set_value(total_days),
                run_time=4,
                rate_func=linear 
            )
            
            # Clean Up Updaters
            window_rect.clear_updaters()
            run_label.clear_updaters()
            partial_curve.clear_updaters()
            
            # Persist Final Curve
            full_curve_static = VMobject().set_points_as_corners(vol_points)
            full_curve_static.set_color(color).set_stroke(width=2)
            final_curves.add(full_curve_static)
            
            # Add static curve, remove dynamic elements
            self.add(full_curve_static)
            self.play(
                FadeOut(window_rect),
                FadeOut(run_label),
                FadeOut(partial_curve),
                run_time=0.5
            )
            self.wait(0.5)
            
        # 6. Final Indication
        self.remove(formula_specific, formula) # Clean old transforms
        formula_final = MathTex(
            r"\sigma_t = \sqrt{\frac{1}{N-1} \sum_{i=0}^{N-1} (r_{t-i} - \bar{r})^2}",
            font_size=32, color=YELLOW
        ).move_to(axes_bot.get_center() + UP * 2.5)
        
        self.play(FadeIn(formula_final))
        self.play(Indicate(formula_final))
        self.wait(2)

    def get_line_graph(self, df, col_name, axes, color, stroke_width=2):
        # Helper to draw static lines
        points = [axes.c2p(i, row[col_name]) for i, row in df.iterrows()]
        line = VMobject().set_points_as_corners(points).set_color(color).set_stroke(width=stroke_width)
        return line

Manim Community v0.19.1

Successfully loaded NVDA data.


<string>:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [11]:
%%manim -v WARNING --fps 120 --disable_caching -qk RSIScene

class RSIScene(Scene):
    def construct(self):
        # 1. Prepare Data
        df_norm = load_data(START_DATE, END_DATE)
        if df_norm is None:
            return

        df = df_norm.rename(columns={'NVDA': 'Price'})
        
        # Calculate RSI
        delta = df['Price'].diff()
        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)
        period = 14
        avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
        avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
        rs = avg_gain / avg_loss
        df['RSI'] = 100 - (100 / (1 + rs))
        
        df = df.dropna().reset_index(drop=True)
        if len(df) > 300:
            df = df.iloc[-300:].reset_index(drop=True)

        price_min, price_max = df['Price'].min(), df['Price'].max()
        p_padding = (price_max - price_min) * 0.1

        # 2. Setup Axes (Note: x_length/y_length used in ManimCE)
        axes_top = Axes(
            x_range=[0, len(df), 30],
            y_range=[price_min - p_padding, price_max + p_padding, (price_max - price_min) / 4],
            x_length=11,
            y_length=3.5,
            axis_config={"color": GREY, "stroke_width": 2, "include_tip": False},
            y_axis_config={"decimal_number_config": {"num_decimal_places": 2}}
        ).to_edge(UP, buff=0.5)
        
        # Add coordinates manually in CE
        axes_top.add_coordinates(font_size=16)

        label_top = Text("Price & Momentum", font_size=24).next_to(axes_top, UP, aligned_edge=LEFT)

        axes_bot = Axes(
            x_range=[0, len(df), 30],
            y_range=[0, 100, 25],
            x_length=11,
            y_length=2.5,
            axis_config={"color": GREY, "stroke_width": 2, "include_tip": False},
            y_axis_config={"decimal_number_config": {"num_decimal_places": 0}}
        ).to_edge(DOWN, buff=0.5)
        
        axes_bot.add_coordinates(font_size=16)

        label_bot = Text("Relative Strength Index (14)", font_size=20, color=PURPLE_B)
        label_bot.next_to(axes_bot, UP, aligned_edge=LEFT)

        self.play(
            Create(axes_top), Write(label_top),
            Create(axes_bot), Write(label_bot)
        )

        # 3. Draw Price Line
        price_line = self.get_line_graph(df, 'Price', axes_top, WHITE, stroke_width=2)
        self.play(Create(price_line), run_time=2.5)

        # 4. RSI Thresholds
        line_70 = DashedLine(axes_bot.c2p(0, 70), axes_bot.c2p(len(df), 70), color=RED, stroke_opacity=0.5)
        line_30 = DashedLine(axes_bot.c2p(0, 30), axes_bot.c2p(len(df), 30), color=GREEN, stroke_opacity=0.5)
        
        txt_70 = Text("Overbought (> 70)", font_size=16, color=RED).next_to(line_70, RIGHT, buff=0.1)
        txt_30 = Text("Oversold (< 30)", font_size=16, color=GREEN).next_to(line_30, RIGHT, buff=0.1)

        self.play(
            Create(line_70), FadeIn(txt_70),
            Create(line_30), FadeIn(txt_30),
            run_time=1.5
        )

        # 5. Draw RSI Curve & Zones
        rsi_points = [axes_bot.c2p(i, val) for i, val in enumerate(df['RSI'])]
        rsi_curve = VMobject().set_points_as_corners(rsi_points).set_color(PURPLE_B).set_stroke(width=2)

        overbought_zones = VGroup()
        oversold_zones = VGroup()

        for i in range(len(df) - 1):
            val = df['RSI'].iloc[i]
            next_val = df['RSI'].iloc[i + 1]
            
            # Check Overbought
            if val > 70 and next_val > 70:
                p1 = axes_bot.c2p(i, val)
                p2 = axes_bot.c2p(i + 1, next_val)
                base1 = axes_bot.c2p(i, 70)
                base2 = axes_bot.c2p(i + 1, 70)
                poly = Polygon(p1, p2, base2, base1, color=RED, fill_opacity=0.3, stroke_width=0)
                overbought_zones.add(poly)
            
            # Check Oversold
            if val < 30 and next_val < 30:
                p1 = axes_bot.c2p(i, val)
                p2 = axes_bot.c2p(i + 1, next_val)
                base1 = axes_bot.c2p(i, 30)
                base2 = axes_bot.c2p(i + 1, 30)
                poly = Polygon(p1, p2, base2, base1, color=GREEN, fill_opacity=0.3, stroke_width=0)
                oversold_zones.add(poly)

        self.play(Create(rsi_curve), run_time=4)
        self.play(FadeIn(overbought_zones), FadeIn(oversold_zones))
        
        # 6. Formula
        formula = MathTex(
            r"RSI = 100 - \frac{100}{1 + \frac{\text{Average Gain}}{\text{Average Loss}}}",
            font_size=36, color=YELLOW
        ).move_to(axes_bot.get_center())
        
        # Add background to make formula readable over the chart
        bg_rect = BackgroundRectangle(formula, color=BLACK, fill_opacity=0.8)

        self.play(FadeIn(bg_rect), FadeIn(formula))
        self.wait(2)
        self.play(FadeOut(bg_rect), FadeOut(formula))
        self.wait(2)

    def get_line_graph(self, df, col_name, axes, color, stroke_width=2):
        points = [axes.c2p(i, row[col_name]) for i, row in df.iterrows()]
        line = VMobject().set_points_as_corners(points).set_color(color).set_stroke(width=stroke_width)
        return line

Manim Community v0.19.1

Successfully loaded NVDA data.
